Pinecone Vector Database with Hugging Face Embeddings
Beginner-Friendly Practical
In this notebook we will learn how to:

Convert text into numerical vectors
Use a Hugging Face embedding model
Create a Pinecone vector database
Store vectors in Pinecone
Architecture
Text ↓ Hugging Face Embedding Model ↓ Vector ↓ Pinecone Vector Database ↓ Similarity Search ↓ Most Relevant Text

Add blockquote

In [1]:
# Install Sentence Transformers.
# This library provides pretrained models that convert text into embeddings.
!pip install -q -U sentence-transformers

# Install the current Pinecone Python SDK.
# This library allows Python to communicate with Pinecone.
!pip install -q -U pinecone

# Install pypdf.
# This library allows us to extract text from PDF documents.
!pip install -q -U pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.6/739.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 2.5 MB/s eta 0:00:00


In [2]:
# Import os.
# We use it to work with environment variables such as API keys.
import os

# Import numpy.
# We use NumPy to work with numerical vectors.
import numpy as np

# Import SentenceTransformer.
# This class loads the Hugging Face embedding model.
from sentence_transformers import SentenceTransformer

# Import Pinecone.
# This class allows Python to connect with Pinecone.
from pinecone import Pinecone

# Import ServerlessSpec.
# This defines the cloud configuration of our Pinecone index.
from pinecone import ServerlessSpec

# Import PdfReader.
# This class allows us to extract text from PDF files.
from pypdf import PdfReader

In [3]:
# Define the Hugging Face embedding model.
# This model converts sentences and paragraphs into numerical vectors.
model_name = "sentence-transformers/all-MiniLM-L6-v2"

# Load the pretrained embedding model.
# The model is downloaded from Hugging Face the first time this cell runs.
embedding_model = SentenceTransformer(model_name)

# Print a message so students know that the model loaded successfully.
print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [4]:
# Create a simple sentence for our first embedding example.
text = "Python is a programming language"

# Convert the text into an embedding vector.
# convert_to_numpy=True returns the result as a NumPy array.
embedding = embedding_model.encode(
    text,
    convert_to_numpy=True
)

# Display the complete embedding vector.
print(embedding)

[-3.53708379e-02  3.81649956e-02 -4.12601344e-02  1.60687584e-02
 -3.83670852e-02 -1.34470552e-01  3.57370339e-02  4.22967821e-02
 -3.46467830e-02 -2.64319927e-02 -4.44902554e-02  2.44822111e-02
  9.47517976e-02  1.70948934e-02  5.69991618e-02 -5.47181740e-02
 -8.70437175e-02 -4.23832098e-03  1.99949145e-02 -1.14943393e-01
 -3.67417969e-02  5.90621270e-02 -3.32976319e-02  2.30448823e-02
  1.57004520e-02 -3.24603659e-03 -9.96119436e-03 -1.88114643e-02
  2.31218711e-02 -2.39675841e-03 -5.23728654e-02  9.51938778e-02
  5.60903996e-02  3.42302471e-02  1.89512931e-02  5.95998354e-02
 -5.56282979e-03 -9.74243507e-02 -5.59518747e-02  3.30421096e-03
 -6.09469824e-02  1.64042730e-02 -5.19629158e-02 -3.67092639e-02
 -5.45357168e-02  5.61590344e-02 -2.10946053e-02 -8.86587799e-03
 -1.90217737e-02 -1.66452229e-02 -1.09764226e-01 -8.97260942e-03
 -3.70126367e-02 -7.99778253e-02 -3.40327783e-03 -1.17237782e-02
  7.25170746e-02  1.41356783e-02 -2.31737196e-02 -1.14860184e-01
 -7.83683509e-02  1.72207

In [5]:
# Create the first sentence.
sentence_1 = "I love programming in Python."

# Create the second sentence.
sentence_2 = "Python is my favorite programming language."

# Convert both sentences into embeddings.
vector_1 = embedding_model.encode(sentence_1, convert_to_numpy=True)
vector_2 = embedding_model.encode(sentence_2, convert_to_numpy=True)

# Print the dimensions of both vectors.
print("Vector 1 dimensions:", len(vector_1))
print("Vector 2 dimensions:", len(vector_2))

Vector 1 dimensions: 384
Vector 2 dimensions: 384


In [6]:
# Import cosine_similarity from scikit-learn.
# It calculates the similarity between two vectors.
from sklearn.metrics.pairwise import cosine_similarity

# Calculate the cosine similarity between our two sentence embeddings.
similarity = cosine_similarity(
    [vector_1],
    [vector_2]
)[0][0]

# Display the similarity score.
print("Similarity Score:", similarity)

Similarity Score: 0.8776499


In [18]:
from google.colab import userdata

# ✅ SECURE: Read from Colab Secrets
PINECONE_API_KEY = userdata.get("PINECONE_API_KEY")

if not PINECONE_API_KEY:
    raise ValueError(
        "PINECONE_API_KEY not found. "
        "Please add it to Colab Secrets."
    )

pc = Pinecone(api_key=PINECONE_API_KEY)
print("Connected to Pinecone successfully!")


Connected to Pinecone successfully!


In [8]:
# Give our Pinecone index a simple name.
index_name = "student-vector-db"

# Get the names of indexes that already exist in our Pinecone account.
existing_indexes = [index.name for index in pc.list_indexes()]

# Check whether our index already exists.
if index_name not in existing_indexes:

    # Create a new Pinecone index.
    pc.create_index(
        # Name of our index.
        name=index_name,

        # The embedding model produces 384-dimensional vectors.
        dimension=384,

        # Cosine similarity is commonly used for semantic search.
        metric="cosine",

        # Define the serverless cloud configuration.
        spec=ServerlessSpec(
            # Cloud provider.
            cloud="aws",

            # Pinecone region.
            region="us-east-1"
        )
    )

    print("Pinecone index created successfully!")

else:

    # If the index already exists, we don't create it again.
    print("Pinecone index already exists.")

Pinecone index already exists.


In [9]:
# Connect to the Pinecone index we created earlier.
index = pc.Index(index_name)

# Print the index object to confirm the connection.
print("Connected to index:", index_name)

Connected to index: student-vector-db


In [19]:
# Create a list of simple documents.
documents = [
    "Python is a popular programming language used for data science and artificial intelligence.",
    "Machine learning allows computers to learn patterns from data.",
    "Deep learning uses neural networks with multiple layers.",
    "Pinecone is a vector database designed for storing and searching embeddings.",
    "Natural language processing allows computers to work with human language."
]

# Print the number of documents.
print("Number of documents:", len(documents))

Number of documents: 5


In [20]:
# Convert all documents into embedding vectors.
# Each document will become a 384-dimensional vector.
document_vectors = embedding_model.encode(
    documents,
    convert_to_numpy=True
)

# Print the shape of the resulting array.
print("Embedding shape:", document_vectors.shape)

Embedding shape: (5, 384)


In [21]:
# Create an empty list where Pinecone records will be stored.
records = []

# Loop through every document and its embedding.
for i, (text, vector) in enumerate(zip(documents, document_vectors)):

    # Create a unique ID for the document.
    record_id = f"doc-{i}"

    # Create a Pinecone record.
    record = {
        # Unique ID of the vector.
        "id": record_id,

        # Convert the NumPy vector into a normal Python list.
        "values": vector.tolist(),

        # Store the original text as metadata.
        "metadata": {
            "text": text
        }
    }

    # Add the record to our list.
    records.append(record)

# Display the first record.
records[0]

{'id': 'doc-0',
 'values': [-0.04477393254637718,
  0.009146657772362232,
  -0.02013455331325531,
  0.053326841443777084,
  -0.023705486208200455,
  -0.1192505955696106,
  0.026245756074786186,
  0.04168618097901344,
  -0.052333079278469086,
  0.011957420967519283,
  -0.05539571866393089,
  0.06770506501197815,
  0.07947323471307755,
  0.02391768991947174,
  0.06970571726560593,
  0.0007716125110164285,
  -0.07678776979446411,
  -0.03556108474731445,
  0.003912458196282387,
  -0.12789468467235565,
  -0.04877806827425957,
  0.05669084191322327,
  -0.0347258560359478,
  -0.01106260996311903,
  0.011783668771386147,
  -0.01596927084028721,
  -0.012049234472215176,
  -0.017667196691036224,
  -0.015180951915681362,
  0.02701796032488346,
  -0.0468127503991127,
  0.06059180945158005,
  0.044195447117090225,
  -0.00689201382920146,
  -0.021830067038536072,
  0.0152933020144701,
  -0.016310401260852814,
  -0.07368107885122299,
  -0.04950574040412903,
  0.03314756602048874,
  -0.055790834128856

In [13]:
# Upload all vector records to Pinecone.
index.upsert(vectors=records)

# Print confirmation.
print("Vectors uploaded to Pinecone successfully!")

Vectors uploaded to Pinecone successfully!



```markdown
## ✅ Summary – What You Built Today

| Step | What You Learned |
|------|------------------|
| **1** | Convert text into numerical **embeddings** using Hugging Face |
| **2** | Generate **384-dimensional vectors** from sentences |
| **3** | Calculate **semantic similarity** between texts |
| **4** | Create a **Pinecone vector database** (cloud-hosted) |
| **5** | Upload vectors with **metadata** (original text) |
| **6** | Store data for **fast similarity search** |

### 🎯 Key Takeaway
```
Text → Embeddings → Vectors → Pinecone → Ready for Semantic Search & RAG
```

### 🚀 What's Next?
- Add **semantic search** queries
- Process **PDF documents** at scale
- Build a **full RAG system** with LLMs

> 💡 **Pro Tip:** Your vector database is now live on Pinecone – you can query it anytime!

---

**🎉 Congratulations! You've built a production-ready vector database.**
```

---

## 📌 **Even Shorter Version (If You Want Ultra-Compact)**

```markdown
## ✅ What You Built

✅ Text → Embeddings (384-dim vectors)  
✅ Stored in Pinecone with metadata  
✅ Ready for semantic search  

**Next:** Add search queries & build a RAG system!

🎉 **Well done!**
```

---
